### Notebook for Cleaning, then splitting into training and testing 

(Before the feature engineering is done)

##### Workflow
1. Load + merge datasets
2. Basic cleaning
3. Split train/test
4. Feature engineering separately on train/test
5. Fit preprocessing on train only
6. Train models
7. Evaluate on test


##### Cleaning includes: 
- datetime conversion
- remove duplicates
- basic NA handling

##### Read in data...

In [1]:
import pandas as pd
import numpy as np

master_df = pd.read_csv("../Data/master_df.csv")

# drops the unnamed col... 
master_df = master_df.drop(columns=['Unnamed: 0'])


# raw_feature_cols = ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 
#                     'order_delivered_carrier_date', 'order_delivered_customer_date', 
#                     'order_estimated_delivery_date', 'customer_zip_code_prefix', 'customer_city', 
#                     'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 
#                     'freight_value', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'product_category_name', 
#                     'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 
#                     'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 
#                     'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'review_id', 
#                     'review_creation_date', 'review_answer_timestamp', 
#                     'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng']


response_cols = ['review_score']

# Removal of textual features? no need to compute sentiment? cus have the review score??
cols_to_remove = ['review_comment_title', 'review_comment_message'] # potentially add more?? the other reviews??  

master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 44 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       119143 non-null  object 
 1   customer_id                    119143 non-null  object 
 2   order_status                   119143 non-null  object 
 3   order_purchase_timestamp       119143 non-null  object 
 4   order_approved_at              118966 non-null  object 
 5   order_delivered_carrier_date   117057 non-null  object 
 6   order_delivered_customer_date  115722 non-null  object 
 7   order_estimated_delivery_date  119143 non-null  object 
 8   customer_unique_id             119143 non-null  object 
 9   customer_zip_code_prefix       119143 non-null  int64  
 10  customer_city                  119143 non-null  object 
 11  customer_state                 119143 non-null  object 
 12  order_item_id                 

##### Intial clean up

In [2]:
master_df['product_name_length']  = master_df['product_name_lenght']        
master_df['product_desc_length']  = master_df['product_description_lenght']

master_df = master_df.drop(columns=['product_description_lenght', 'product_name_lenght'])

##### Convert into datetime...

In [3]:
date_time_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date',
    'review_answer_timestamp',
]
for col in date_time_cols:
    master_df[col] = pd.to_datetime(master_df[col], errors='coerce')

master_df = master_df.dropna(subset=['review_creation_date'])

##### Dealing with NAs

- Review score (if na then drop...) no response no ...
- Review_comment_title, review_comment_message --> Fill with '' (if missing just means that no review comment given)
- Drop everything that wasn't delivered... no response??
- payment_type if not defined... drop...
- seller_id, product_id, order_item_id
    - Drop rows where these are missing.

- Product category columns:
    - product_category_name
    - product_category_name_english
    - Fill missing values with unknown.

- customer_lat, customer_lng, seller_lat, seller_lng
    - Drop rows where these are missing.
    - Reason: required to compute delivery distance between customer and seller.

- order_approved_at, order_delivered_carrier_date, order_delivered_customer_date
    - Drop rows where these are missing.
    - Reason: these timestamps are required for constructing delivery lifecycle features (e.g. shipping delay, delivery delay).

- payment_sequential, payment_value, payment_installments, payment_type
    - Drop rows where these are missing.
    - Reason: payment information is critical for financial behavioural features, and missing values represent incomplete transaction records.

- product_weight_g, product_length_cm, product_height_cm, product_width_cm
    - Fill missing values using the mean within each product category.
    - Reason: product size depends on category, so category-level imputation preserves structure better than global imputation.

- product_name_length, product_desc_length, product_photos_qty
    - Fill missing values using the mean within each product category.
    - Reason: product listing metadata varies by category, so category-level averages provide a more realistic estimate than global values.

In [4]:
master_df.isnull().sum()

order_id                              0
customer_id                           0
order_status                          0
order_purchase_timestamp              0
order_approved_at                   172
order_delivered_carrier_date       2048
order_delivered_customer_date      3285
order_estimated_delivery_date         0
customer_unique_id                    0
customer_zip_code_prefix              0
customer_city                         0
customer_state                        0
order_item_id                       814
product_id                          814
seller_id                           814
shipping_limit_date                 814
price                               814
freight_value                       814
seller_zip_code_prefix              814
seller_city                         814
seller_state                        814
product_category_name              2509
product_photos_qty                 2509
product_weight_g                    834
product_length_cm                   834


In [ ]:
master_df = master_df[master_df['review_score'].notna()].copy()
master_df = master_df[master_df['order_status'] == 'delivered'].copy()
master_df = master_df[master_df['seller_id'].notna()].copy()

# Required cols...
required_cols = [
    'seller_id',
    'product_id',
    'order_item_id'
]

master_df = master_df.dropna(subset=required_cols)

master_df['review_comment_title'] = (
    master_df['review_comment_title']
    .fillna('')
)

master_df['review_comment_message'] = (
    master_df['review_comment_message']
    .fillna('')
)

master_df['product_category_name'] = (
    master_df['product_category_name']
    .fillna('unknown')
)

master_df['product_category_name_english'] = (
    master_df['product_category_name_english']
    .fillna('unknown')
)

# Just drop... not a big amount 
master_df = master_df.dropna(
    subset=[
        'customer_lat',
        'customer_lng',
        'seller_lat',
        'seller_lng'
    ]
)

# Just drop... not a big amount 
master_df = master_df.dropna(
    subset=[
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
    ]
)

# Just drop... not a big amount (only 3)
master_df = master_df.dropna(
    subset=[
        'payment_sequential',
        'payment_value',
        'payment_installments',
        'payment_type',
    ]
)

# dimension cols... compute the mean of the category name it they are in... 
dim_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for col in dim_cols:
    master_df[col] = master_df.groupby(
        'product_category_name_english'
    )[col].transform(lambda x: x.fillna(x.mean()))


# product weight etc cols... compute the mean of the category name it they are in... too 
meta_cols = [
    'product_name_length',
    'product_desc_length',
    'product_photos_qty'
]

for col in meta_cols:
    master_df[col] = master_df.groupby(
        'product_category_name_english'
    )[col].transform(lambda x: x.fillna(x.mean()))


##### Verifying that no nas left

In [39]:
master_df.isnull().sum()

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
customer_unique_id               0
customer_zip_code_prefix         0
customer_city                    0
customer_state                   0
order_item_id                    0
product_id                       0
seller_id                        0
shipping_limit_date              0
price                            0
freight_value                    0
seller_zip_code_prefix           0
seller_city                      0
seller_state                     0
product_category_name            0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
payment_sequential  

##### Duplicates handling

- Initial Duplicates handling...
- Duplicates by order id...

In [40]:
master_df = master_df.drop_duplicates()

##### Order duplication handling

In [41]:
print(master_df['order_id'].nunique())
len(master_df)


95333


114273

##### Split into training and testing data before feature engineering

Causes are:
- payment rows (mulitple payment methods...) --> turn into binary indicator cols
- cus multiple payment methods, multiple installments / prices, so aggregate that

In [42]:
# # Remove the display limit for columns
# pd.set_option('display.max_columns', None)

# # Optional: if rows are very long, prevent wrapping
# pd.set_option('display.expand_frame_repr', False)

# # Now print your duplicates
# print(master_df[master_df.duplicated(subset=['order_id'], keep=False)])

In [43]:
# Step 1: payment dummies FIRST (before groupby collapses the rows)
payment_dummies = pd.get_dummies(
    master_df[['order_id', 'payment_type']],
    columns=['payment_type'],
    prefix='paid'
)
payment_dummies = payment_dummies.groupby('order_id').max().reset_index()

# Step 2: main aggregation
agg_dict = {
    # order-level — constant within an order
    'customer_id':                   'first',
    'customer_unique_id':            'first',
    'order_status':                  'first',
    'order_purchase_timestamp':      'first',
    'order_approved_at':             'first',
    'order_delivered_carrier_date':  'first',
    'order_delivered_customer_date': 'first',
    'order_estimated_delivery_date': 'first',
    'customer_zip_code_prefix':      'first',
    'customer_city':                 'first',
    'customer_state':                'first',
    'customer_lat':                  'first',
    'customer_lng':                  'first',
    'shipping_limit_date':           'min',

    # item-level
    'order_item_id':                 'max',    # max = num items
    'price':                         'sum',    # total order price
    'freight_value':                 'sum',    # total freight
    'product_id':                    'first',
    'seller_id':                     'first',
    'seller_zip_code_prefix':        'first',
    'seller_city':                   'first',
    'seller_state':                  'first',
    'seller_lat':                    'first',
    'seller_lng':                    'first',

    # product dims — mean across items
    'product_category_name':         lambda x: x.mode().iloc[0],
    'product_category_name_english': lambda x: x.mode().iloc[0],
    'product_photos_qty':            'mean',
    'product_weight_g':              'mean',
    'product_length_cm':             'mean',
    'product_height_cm':             'mean',
    'product_width_cm':              'mean',
    'product_name_length':           'mean',
    'product_desc_length':           'mean',

    # payment — no payment_type here, handled by dummies above
    'payment_sequential':            'max',    # num payment methods
    'payment_installments':          'max',    # credit card installments
    'payment_value':                 'sum',    # sum across split payments

    # review — 1:1 with order
    'review_id':                     'first',
    'review_score':                  'first',
    'review_creation_date':          lambda x: x.dropna().iloc[0] if x.notna().any() else pd.NaT,
    'review_answer_timestamp':       'first',
    'review_comment_title':          'first',
    'review_comment_message':        'first',
}

master_df_agg = master_df.groupby('order_id').agg(agg_dict).reset_index()

# Step 3: rename to make cols self-explanatory
master_df_agg = master_df_agg.rename(columns={
    'order_item_id':      'num_items',
    'price':              'total_price',
    'freight_value':      'total_freight',
    'payment_sequential': 'num_payment_methods',
})

# Step 4: merge payment binary indicator in 
master_df = master_df_agg.merge(payment_dummies, on='order_id', how='left')


In [44]:
print(master_df['order_id'].nunique())
len(master_df)


95333


95333

In [45]:
master_df.isnull().sum()


order_id                         0
customer_id                      0
customer_unique_id               0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
customer_zip_code_prefix         0
customer_city                    0
customer_state                   0
customer_lat                     0
customer_lng                     0
shipping_limit_date              0
num_items                        0
total_price                      0
total_freight                    0
product_id                       0
seller_id                        0
seller_zip_code_prefix           0
seller_city                      0
seller_state                     0
seller_lat                       0
seller_lng                       0
product_category_name            0
product_category_name_english    0
product_photos_qty               0
product_weight_g    

In [46]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95333 entries, 0 to 95332
Data columns (total 47 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       95333 non-null  object        
 1   customer_id                    95333 non-null  object        
 2   customer_unique_id             95333 non-null  object        
 3   order_status                   95333 non-null  object        
 4   order_purchase_timestamp       95333 non-null  datetime64[ns]
 5   order_approved_at              95333 non-null  datetime64[ns]
 6   order_delivered_carrier_date   95333 non-null  datetime64[ns]
 7   order_delivered_customer_date  95333 non-null  datetime64[ns]
 8   order_estimated_delivery_date  95333 non-null  datetime64[ns]
 9   customer_zip_code_prefix       95333 non-null  int64         
 10  customer_city                  95333 non-null  object        
 11  customer_state 

##### Dtypes Validation

- Conversion to Ints
- Categories too 
- nominal etc

In [47]:
int_cols = [
    'num_items',          # position counter 1,2,3...
    'product_photos_qty',     # count of photos
    'product_weight_g',       # grams — always whole numbers in this dataset
    'product_length_cm',      # dimensions — whole numbers
    'product_height_cm',
    'product_width_cm',
    'num_payment_methods',     # 1,2,3... counter
    'payment_installments',   # number of installments
    'review_score',           # 1–5 rating
    'customer_zip_code_prefix',  # already int64, but worth keeping consistent
    'seller_zip_code_prefix', # stored as float due to NAs during cleaning
    'product_name_length',    # character count
    'product_desc_length',    # character count
]

master_df[int_cols] = master_df[int_cols].astype(int)

In [48]:
# Nominal categories — no natural order
nominal_cols = [
    'order_status',           # only 'delivered' at this point anyway
    'customer_city',
    'customer_state',         
    'seller_city',
    'seller_state',
    'paid_boleto',
    'paid_credit_card',                      
    'paid_debit_card',                     
    'paid_voucher',
    'product_category_name',
    'product_category_name_english',
]

for col in nominal_cols:
    master_df[col] = master_df[col].astype('category')

# Ordinal — review score has a natural order (1 < 2 < 3 < 4 < 5)
master_df['review_score'] = pd.Categorical(
    master_df['review_score'],
    categories=[1, 2, 3, 4, 5],
    ordered=True
)

##### Drop not needed ID cols

- keeping order ids... unqiue on this 
- keeping unique customer id... for repeat customers... 

In [49]:
drop_id_cols = [
    'customer_id',          # order-scoped session key — useless after agg
    'product_id',           # now captured by num_unique_products          
    'review_id',            # just a UUID, zero signal
]

master_df = master_df.drop(columns=drop_id_cols)

In [50]:
master_df.columns.tolist()

['order_id',
 'customer_unique_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'customer_lat',
 'customer_lng',
 'shipping_limit_date',
 'num_items',
 'total_price',
 'total_freight',
 'seller_id',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'seller_lat',
 'seller_lng',
 'product_category_name',
 'product_category_name_english',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'product_name_length',
 'product_desc_length',
 'num_payment_methods',
 'payment_installments',
 'payment_value',
 'review_score',
 'review_creation_date',
 'review_answer_timestamp',
 'review_comment_title',
 'review_comment_message',
 'paid_boleto',
 'paid_credit_card',
 'paid_debit_card',
 'paid_voucher']

##### Save the Cleaned dataset

In [51]:
master_df.to_csv("../Data/cleaned_master_df.csv")

In [55]:
master_df["review_creation_date"].isna().sum()

0

In [11]:
df = pd.read_csv("../Data/pre_split_features.csv")

print(df.isnull().sum().to_list())

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
